# Lab 25-09 · Extrapolating Reliability

**Predict first**, then run. The handout is [README.md](README.md). Part 4 (ngspice) runs from the command line: `ngspice -b hbm_esd.cir`.

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize, stats
import reliability as r

## Part 1 · Diverging models

*Predict* the field at which the E and power-law extrapolations differ by a factor of 1,000.

In [2]:
f = r.tddb_fit(10, 10, 9, 100)
print("gamma, G, m =", [round(x, 3) for x in f["params"]])
e_k = optimize.brentq(lambda e: f["power"](e) / f["E"](e) - 1000, 1, 9.5)
print(f"factor of 1000 at {e_k:.2f} MV/cm")
e = np.linspace(2.5, 10, 100)
plt.figure()
for name in ("E", "power", "1/E"):
    plt.semilogy(e, [f[name](x) for x in e], label=name)
plt.xlabel("field (MV/cm)"); plt.ylabel("characteristic life (s)"); plt.legend(); plt.grid(True)
plt.show()

gamma, G, m = [2.303, 207.233, 21.854]
factor of 1000 at 3.79 MV/cm


## Part 2 · Weibull slope

*Predict* the 100 ppm scale factor at β = 1.2.

In [3]:
print(f"beta 2: {r.weibull_scale(1000, 1e-4, 2):.3g}; beta 1.2: {r.weibull_scale(1000, 1e-4, 1.2):.3g}")
b = np.linspace(0.8, 4, 60)
plt.figure()
plt.semilogy(b, [r.weibull_scale(1000, 1e-4, x) for x in b])
plt.xlabel("Weibull shape beta"); plt.ylabel("chip 100 ppm life / test t63"); plt.grid(True)
plt.show()

beta 2: 0.000316; beta 1.2: 1.47e-06


## Part 3 · What 60% means

*Predict* the fraction of zero-failure runs for a product whose true rate equals the bound.

In [4]:
bound = r.fit_zero_fail(7859, 1000, 78.6)
p = r.simulate_htol(7859, 1000, 78.6, bound)
print(f"bound {bound:.3f} FIT; simulated pass fraction {p:.4f}; exact {math.exp(-stats.chi2.ppf(0.6, 2) / 2):.4f}")
fr = np.linspace(0.1, 3, 40)
plt.figure()
plt.plot(fr, [math.exp(-x * stats.chi2.ppf(0.6, 2) / 2) for x in fr])
plt.xlabel("true rate / 60% bound"); plt.ylabel("probability of zero failures"); plt.grid(True)
plt.show()

bound 1.483 FIT; simulated pass fraction 0.3991; exact 0.4000


## Part 4 · ESD clamp

The ngspice netlist checks the 1 Ω clamp; here is the formula for the 2 Ω case to compare with your rerun.

In [5]:
print(r.hbm(2000, 5, 1))
print(r.hbm(2000, 5, 2))

{'i_peak': 1.3291139240506329, 'tau': 1.501e-07, 'v_pad': 6.329113924050633, 'energy': 0.0002}
{'i_peak': 1.3282290279627165, 'tau': 1.502e-07, 'v_pad': 7.656458055925433, 'energy': 0.0002}
